## 1. Lendo o arquivo fonte

Nesta prática, vamos usar uma página fictícia do livro `Integrações Resilientes`. A fonte não começa como chunk: primeiro lemos o arquivo original e depois escolhemos como quebrá-lo.

In [1]:
from pathlib import Path

from chunking_common import parse_frontmatter

DATA_PATH = Path("../data/integracoes-resilientes-webhooks.md")
source_document = DATA_PATH.read_text(encoding="utf-8")

print(f"arquivo fonte: {DATA_PATH}")
print(f"tamanho do arquivo: {len(source_document)} caracteres")

arquivo fonte: ../data/integracoes-resilientes-webhooks.md
tamanho do arquivo: 1589 caracteres


## 2. Separando metadados e texto bruto

O frontmatter descreve a origem da página inteira. Vamos separá-lo do corpo para que os metadados viajem com cada chunk, sem serem tratados como texto do livro.

In [2]:
source_page = parse_frontmatter(source_document)
metadata = source_page.metadata

print(f"livro: {metadata.book_title}")
print(f"trecho: {metadata.chapter} > {metadata.section}")
print(f"páginas: {metadata.page_start}-{metadata.page_end}")
print(f"texto bruto: {len(source_page.text)} caracteres")

livro: Integrações Resilientes: webhooks, filas e retentativas na prática
trecho: Capítulo 4 - Webhooks em produção > 4.3 Timeouts e retentativas
páginas: 118-119
texto bruto: 1363 caracteres


## 3. Definindo tamanho e overlap

Nesta primeira estratégia, não vamos carregar um tokenizer real. Vamos usar uma estimativa simples, de aproximadamente 1 token a cada 4 caracteres, para controlar o tamanho sem esconder que a conta é aproximada.

In [3]:
from dataclasses import dataclass
import math


@dataclass(frozen=True)
class FixedSizeConfig:
    max_chars: int
    overlap_chars: int

    def __post_init__(self):
        if self.max_chars <= 0:
            raise ValueError("max_chars precisa ser maior que zero.")
        if self.overlap_chars < 0 or self.overlap_chars >= self.max_chars:
            raise ValueError("overlap_chars precisa estar entre zero e max_chars.")


CHARS_PER_ESTIMATED_TOKEN = 4
config = FixedSizeConfig(max_chars=720, overlap_chars=160)


def estimate_tokens(text):
    return math.ceil(len(text) / CHARS_PER_ESTIMATED_TOKEN)


print("configuração:")
print(f"  tamanho máximo: {config.max_chars} caracteres")
print(f"  overlap: {config.overlap_chars} caracteres")
print(f"  estimativa: 1 token ~= {CHARS_PER_ESTIMATED_TOKEN} caracteres")
print(f"  texto bruto: ~{estimate_tokens(source_page.text)} tokens")

configuração:
  tamanho máximo: 720 caracteres
  overlap: 160 caracteres
  estimativa: 1 token ~= 4 caracteres
  texto bruto: ~341 tokens


## 4. Criando chunks por tamanho fixo

O algoritmo abaixo conhece apenas o texto, o tamanho máximo e o overlap. Ele não sabe onde existe uma tabela, um payload ou uma fronteira de sentido. O recorte preserva os offsets originais para que a posição do chunk continue verdadeira.

In [4]:
@dataclass(frozen=True)
class FixedChunk:
    chunk_id: str
    text: str
    char_start: int
    char_end: int

    @property
    def char_count(self):
        return self.char_end - self.char_start

    @property
    def estimated_tokens(self):
        return estimate_tokens(self.text)


def chunk_by_fixed_size(text, config):
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + config.max_chars, len(text))
        chunks.append(FixedChunk(
            chunk_id=f"chunk-fixed-{len(chunks) + 1:02d}",
            text=text[start:end],
            char_start=start,
            char_end=end,
        ))

        if end == len(text):
            break
        start = end - config.overlap_chars

    return tuple(chunks)


chunks = chunk_by_fixed_size(source_page.text, config)

print(f"chunks criados: {len(chunks)}")
print("id               início   fim    caracteres   tokens estimados")
print("-----------------|--------|------|------------|-----------------")
for chunk in chunks:
    print(
        f"{chunk.chunk_id:<17} {chunk.char_start:>6}   {chunk.char_end:>4} "
        f"   {chunk.char_count:>6}       ~{chunk.estimated_tokens:>4}"
    )

chunks criados: 3
id               início   fim    caracteres   tokens estimados
-----------------|--------|------|------------|-----------------
chunk-fixed-01         0    720       720       ~ 180
chunk-fixed-02       560   1280       720       ~ 180
chunk-fixed-03      1120   1363       243       ~  61


## 5. Inspecionando as fronteiras

Em vez de imprimir todos os chunks como uma sequência de strings, vamos olhar para as fronteiras. A saída mostra o fim de um chunk, o início do próximo e o trecho repetido pelo overlap.

In [5]:
from itertools import pairwise

from chunking_common import compact_preview


for previous, current in pairwise(chunks):
    overlap = source_page.text[current.char_start:previous.char_end]
    print(f"CORTE: {previous.chunk_id} -> {current.chunk_id}")
    print(f"fim do chunk anterior: {compact_preview(previous.text[-80:], max_chars=80)}")
    print(f"início do próximo: {compact_preview(current.text[:80], max_chars=80)}")
    print(f"texto repetido pelo overlap ({len(overlap)} caracteres):")
    print(compact_preview(overlap, max_chars=120))
    print()

CORTE: chunk-fixed-01 -> chunk-fixed-02
fim do chunk anterior: | 200 | evento recebido e persistido | encerrar entrega | | 408 |
consumidor não
início do próximo: no consumidor. | status | significado | ação recomendada | | --- | --- |
--- |
texto repetido pelo overlap (160 caracteres):
no consumidor. | status | significado | ação recomendada | | --- | --- |
--- | | 200 | evento recebido e persistido | en...

CORTE: chunk-fixed-02 -> chunk-fixed-03
fim do chunk anterior: síveis. Se a mesma entrega aparecer de novo depois de um timeout, a
aplicação re
início do próximo: 0Z" } ``` O consumidor deve gravar `event_id` antes de executar efeitos
irrever
texto repetido pelo overlap (160 caracteres):
0Z" } ``` O consumidor deve gravar `event_id` antes de executar efeitos
irreversíveis. Se a mesma entrega aparecer de no...



## 6. Anexando metadados

Os metadados globais identificam a origem. A posição e o tamanho são específicos de cada chunk.

In [6]:
metadata_chunks = []
for chunk in chunks:
    metadata_chunks.append({
        "text": chunk.text,
        "metadata": {
            "book_title": metadata.book_title,
            "edition": metadata.edition,
            "chapter": metadata.chapter,
            "section": metadata.section,
            "page_start": metadata.page_start,
            "page_end": metadata.page_end,
            "chunk_id": chunk.chunk_id,
            "strategy": "fixed_size_with_overlap",
            "char_start": chunk.char_start,
            "char_end": chunk.char_end,
            "estimated_tokens": chunk.estimated_tokens,
            "source_file": str(DATA_PATH),
        },
    })

print(f"metadados comuns: {metadata.book_title} | {metadata.edition} | páginas {metadata.page_start}-{metadata.page_end}")
print("id               estratégia                 posição       tokens")
print("-----------------|--------------------------|--------------|-------")
for item in metadata_chunks:
    item_metadata = item["metadata"]
    position = f"{item_metadata['char_start']}-{item_metadata['char_end']}"
    print(
        f"{item_metadata['chunk_id']:<17} {item_metadata['strategy']:<26} "
        f"{position:>12}   ~{item_metadata['estimated_tokens']:>4}"
    )

metadados comuns: Integrações Resilientes: webhooks, filas e retentativas na prática | 2ª edição | páginas 118-119
id               estratégia                 posição       tokens
-----------------|--------------------------|--------------|-------
chunk-fixed-01    fixed_size_with_overlap           0-720   ~ 180
chunk-fixed-02    fixed_size_with_overlap        560-1280   ~ 180
chunk-fixed-03    fixed_size_with_overlap       1120-1363   ~  61


## 7. Recuperando candidatos

A busca abaixo conta termos da query nos chunks. Ela é propositalmente simples: queremos observar qual unidade o corte fixo oferece, não reensinar ranking lexical.

In [7]:
from chunking_common import print_search_summary, search


query = "como lidar com timeout em webhooks?"
results = search(query, metadata_chunks)
print_search_summary(query, results)

query: como lidar com timeout em webhooks?
resultados por contagem simples de termos:
posição | chunk                  | score | termos
--------|------------------------|-------|----------------
      1 | chunk-fixed-01         |     2 | timeout, webhooks
      2 | chunk-fixed-02         |     1 | timeout
      3 | chunk-fixed-03         |     1 | timeout

maior score simplificado: 2
candidato(s) no topo: chunk-fixed-01
prévia do primeiro candidato no topo:
# 4.3 Timeouts e retentativas em webhooks Quando um provedor envia um
webhook, ele espera uma resposta rápida do consumidor. Se a conexão
expira antes de receber confirmação, o pro...


O tamanho fixo produziu chunks previsíveis, mas as fronteiras atravessaram unidades importantes. O overlap ficou visível como texto repetido entre chunks vizinhos; ele reduz parte da perda de contexto, mas não entende o documento.